# AICoverGen Remaster + Applio (WebUI)

Sube `aicovergen_remaster.zip` y ejecuta la celda de abajo para lanzar la interfaz completa.


In [ ]:
# @title 🚀 Lanzar AICoverGen WebUI
import os, zipfile, shutil, sys, subprocess, requests, time
from google.colab import files
from pathlib import Path

PROJECT_DIR = Path('/content/aicovergen_remaster')

# 1. Preparar Proyecto
if not PROJECT_DIR.exists():
    print("Sube aicovergen_remaster.zip...")
    uploaded = files.upload()
    zip_name = next((n for n in uploaded if n.lower().endswith('.zip')), None)
    if not zip_name: raise RuntimeError("No se subió el ZIP.")
    
    print("Extrayendo...")
    with zipfile.ZipFile(zip_name, 'r') as z: z.extractall('/content/_tmp')
    root = next(Path('/content/_tmp').rglob('README_REMASTER.md')).parent
    shutil.move(str(root), str(PROJECT_DIR))
    shutil.rmtree('/content/_tmp')
    os.remove(zip_name)

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR / 'src') not in sys.path: sys.path.insert(0, str(PROJECT_DIR / 'src'))
if str(PROJECT_DIR) not in sys.path: sys.path.insert(0, str(PROJECT_DIR))

# 2. Instalar Dependencias
try:
    import gradio, pedalboard
    print("Dependencias ya instaladas.")
except:
    print("Instalando dependencias (esto tarda 2-3 min)...")
    !apt-get update -qq && apt-get install -y -qq ffmpeg sox libsox-dev libsndfile1
    !pip install -q -r requirements_colab.txt

# 3. Descargar Modelos Base (MDX corregido con curl y verificación)
from rvc.lib.tools.prerequisites_download import prequisites_download_pipeline
prequisites_download_pipeline(pretraineds_hifigan=False, models=True, exe=False)

mdx_dir = PROJECT_DIR / 'mdxnet_models'
mdx_dir.mkdir(exist_ok=True)
mdx_urls = {
    'UVR-MDX-NET-Voc_FT.onnx': 'https://huggingface.co/Politrees/UVR_resources/resolve/main/MDXNet_models/UVR-MDX-NET-Voc_FT.onnx',
    'UVR_MDXNET_KARA_2.onnx': 'https://huggingface.co/Politrees/UVR_resources/resolve/main/MDXNet_models/UVR_MDXNET_KARA_2.onnx',
    'Reverb_HQ_By_FoxJoy.onnx': 'https://huggingface.co/Politrees/UVR_resources/resolve/main/MDXNet_models/Reverb_HQ_By_FoxJoy.onnx',
    'UVR-MDX-NET-Inst_HQ_4.onnx': 'https://huggingface.co/Politrees/UVR_resources/resolve/main/MDXNet_models/UVR-MDX-NET-Inst_HQ_4.onnx',
}
for name, url in mdx_urls.items():
    dest = mdx_dir / name
    # Si el archivo no existe o es demasiado pequeño (corrupto), descargar
    if not dest.exists() or dest.stat().st_size < 10000000: 
        print(f"Descargando {name} (esto puede tardar)...")
        if dest.exists(): dest.unlink()
        !curl -L -o "{dest}" "{url}"

# 4. Lanzar WebUI
print("\nLanzando interfaz... Haz clic en el enlace 'public URL' cuando aparezca.")
os.environ['PYTHONPATH'] = f"{PROJECT_DIR}:{PROJECT_DIR}/src:{os.environ.get('PYTHONPATH', '')}"
!python src/webui.py --share
